# Analysis of Infobae news with NLP

**Question:** what can be learned from a corpus of Spanish-language news using natural language processing techniques, and how do those techniques compare with each other?

**Data:** articles published in the Infobae RSS feed, accumulated with `src/actualizar_corpus.py`. The feed only delivers the latest ~100 articles per query, so the corpus is built by running the script at different times.

**Note on the data:** the corpus contains the full text of the articles, which is copyrighted, so it is not included in the repository. To rebuild it, run the script.

In [24]:
# The libraries may take a minute to load.
# Standard library
import time
from collections import Counter, defaultdict
from pathlib import Path

# Data
import pandas as pd

# PLN
import spacy

#Libraries associated to ML
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#Libraries associated to BERTopic
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

from sentence_transformers import SentenceTransformer, util
SEED = 42    # fixed seed so that the results can be reproduced

In [25]:
RAIZ = Path.cwd().parent          # this notebook is in notebooks/
d = pd.read_csv(RAIZ / "data" / "noticias_infobae.csv")
n_antes = len(d)

d = d.dropna(subset=["texto"]).reset_index(drop=True)
print(f" {n_antes - len(d)} articles without text were discarded")

d["fecha"] = pd.to_datetime(d["fecha"], utc=True)
d["section"] = d["url"].str.split("/").str[3]
d["words"] = d["texto"].str.split().str.len()

print(d.shape)
print(d["fecha"].min(), "->", d["fecha"].max())
print(d["section"].value_counts())
print(d["words"].describe().round(0))

 3 articles without text were discarded
(2500, 7)
2026-10-23 19:00:54+00:00 -> 2026-10-27 03:27:26+00:00
section
america                    775
espana                     428
mexico                     277
colombia                   193
peru                       179
deportes                    62
tecno                       58
salud                       50
teleshow                    49
sociedad                    41
politica                    39
economia                    33
estados-unidos              26
noticias                    25
tendencias                  25
judiciales                  21
el-salvador                 19
panama                      19
entretenimiento             16
honduras                    16
cultura                     15
opinion                     15
malditos-nerds              14
cuba                        14
costa-rica                  13
guatemala                   12
nicaragua                   11
republica-dominicana         8
historias          

## 2. Preprocessing

The text is cleaned with spaCy: lowercasing, tokenization, stopword removal, and lemmatization. Negations (`no`, `nunca`(never), `sin`(without)...) are kept, because spaCy treats them as stopwords and discarding them reverses the meaning of phrases like "no me gustó".

This cleaned text is used with TF-IDF. The embedding and transformer models receive the original text, because they are trained on complete sentences.

In [26]:
nlp = spacy.load("es_core_news_md", disable=["parser", "ner"])
negaciones = {"no", "nunca", "ni", "sin", "jamás", "tampoco"}


def limpiar(texto):
    doc = nlp(texto.lower())
    tokens = [t.lemma_.lower() for t in doc
              if (not t.is_stop or t.lower_ in negaciones) and t.is_alpha]
    return " ".join(tokens)


t0 = time.time()
d["limpio"] = d["texto"].apply(limpiar)


## 3. Predicting the newspaper edition

**Task:** given the text of an article, predict whether it belongs to the `america`, `espana`(Spain), `mexico`, `peru`, or `colombia` edition (the edition comes from the URL). Only these five are used because the rest of the sections have very few articles.

**Clarification:** the edition is not a topic. A sports article from Mexico and a politics article from Mexico have the same label. The task measures whether the vocabulary of an article gives away its origin.

**Evaluation:** stratified 5-fold cross-validation, with macro F1 (averages the classes equally) and compared with a baseline that predicts the same edition (the most frequent one) for every article.

### 3.1 Baseline: TF-IDF + logistic regression

In [27]:
EDICIONES = ["america", "espana", "mexico", "peru", "colombia"]
mask = d["section"].isin(EDICIONES)
d5 = d[mask]
y = d5["section"]

baseline = y.value_counts(normalize=True).max()
print(f"{len(d5)} articles | baseline (most frequent): accuracy {baseline:.2f} ")

pipe_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=2, max_df=0.5)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", C=10)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pred_tfidf = cross_val_predict(pipe_tfidf, d5["limpio"], y, cv=cv)

print("F1 macro:", round(f1_score(y, pred_tfidf, average="macro"), 2))
print(classification_report(y, pred_tfidf))

etiq = sorted(EDICIONES)
print(pd.DataFrame(confusion_matrix(y, pred_tfidf, labels=etiq),
                   index=[f"real_{e}" for e in etiq],
                   columns=[f"pred_{e}" for e in etiq]))

1852 articles | baseline (most frequent): accuracy 0.42 
F1 macro: 0.85
              precision    recall  f1-score   support

     america       0.85      0.79      0.82       775
    colombia       0.92      0.91      0.91       193
      espana       0.71      0.79      0.75       428
      mexico       0.88      0.94      0.91       277
        peru       0.91      0.86      0.89       179

    accuracy                           0.83      1852
   macro avg       0.85      0.86      0.85      1852
weighted avg       0.83      0.83      0.83      1852

               pred_america  pred_colombia  pred_espana  pred_mexico  \
real_america            612             10          121           21   
real_colombia             5            175            5            7   
real_espana              82              0          339            5   
real_mexico              11              1            5          259   
real_peru                13              5            5            2   

      

### 3.2 Embeddings + logistic regression

Each article is represented with a 384-dimensional embedding (`paraphrase-multilingual-MiniLM-L12-v2`), computed on the original text, and the same classifier is trained with the same folds. The model reads only the first ~128 tokens of each article.

In [28]:
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
E = modelo_emb.encode(d["texto"].tolist(), show_progress_bar=True)
print(E.shape)

E5 = E[mask.to_numpy()]
clf_emb = LogisticRegression(max_iter=2000, class_weight="balanced", C=10)
pred_emb = cross_val_predict(clf_emb, E5, y, cv=cv)

print("F1 macro TF-IDF:      ", round(f1_score(y, pred_tfidf, average="macro"), 2))
print("F1 macro embeddings:  ", round(f1_score(y, pred_emb, average="macro"), 2))
print(classification_report(y, pred_emb))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

(2500, 384)
F1 macro TF-IDF:       0.85
F1 macro embeddings:   0.65
              precision    recall  f1-score   support

     america       0.80      0.69      0.74       775
    colombia       0.55      0.65      0.60       193
      espana       0.66      0.69      0.68       428
      mexico       0.64      0.70      0.67       277
        peru       0.54      0.61      0.57       179

    accuracy                           0.68      1852
   macro avg       0.64      0.67      0.65      1852
weighted avg       0.69      0.68      0.68      1852



**Control:** TF-IDF reads the full article and the embedding model only the first ~128 tokens. To compare on equal terms, TF-IDF is repeated with each article trimmed to its first 90 words.

In [29]:
d["texto_corto"] = d["texto"].str.split().str[:90].str.join(" ")
d["limpio_corto"] = d["texto_corto"].apply(limpiar)

pred_tfidf_corto = cross_val_predict(pipe_tfidf, d.loc[mask, "limpio_corto"], y, cv=cv)

print("F1 macro TF-IDF (complete text):    ", round(f1_score(y, pred_tfidf, average="macro"), 2))
print("F1 macro TF-IDF (first 90 words):  ", round(f1_score(y, pred_tfidf_corto, average="macro"), 2))
print("F1 macro embeddings:                 ", round(f1_score(y, pred_emb, average="macro"), 2))

F1 macro TF-IDF (complete text):     0.85
F1 macro TF-IDF (first 90 words):   0.79
F1 macro embeddings:                  0.65


### 3.3 Conclusion

| Method | Text it reads | Macro F1 |
|---|---|---|
| Baseline (always `america`) | - | accuracy 0.42 |
| TF-IDF + logistic regression | full article | 0.85 |
| TF-IDF + logistic regression | first 90 words | 0.79 |
| Embeddings + logistic regression | first ~128 tokens | 0.65 |

- TF-IDF outperforms embeddings by 14 points with text of comparable length; another 6 points of advantage come from reading the full article.
- **Hypothesis (not verified):** the edition is recognized by specific words (places, politicians, institutions) that TF-IDF sees directly, whereas an embedding summarizes the topic.
- The main error is the confusion between `america` and `espana`: 203 of the 313 errors (65%). It could be explored whether this is because `america` groups agency wire stories and general topics.
  It could be explored whether this is because `america` groups agency wire stories and general topics.

**Limitations:** the corpus covers about 3 days, so the same news story may appear in several articles and be split between training and test, which inflates the result. The hyperparameters (`C=10`, `class_weight="balanced"`) were chosen by looking at previous results.

## 4. Discovering topics without labels

Articles are grouped by similarity of meaning, using the embeddings, without any labels. First with KMeans (the number of groups has to be chosen) and then with BERTopic (it chooses it on its own).

### 4.1 KMeans: choosing the number of groups

The silhouette measures how well separated the groups are (closer to 1 is better). With text it usually gives low values, so it is useful for comparing between values of `k`. The final decision is made by reading the groups.

In [30]:
for k in [4, 6, 8, 10, 12, 15]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    etiquetas = km.fit_predict(E)
    sil = silhouette_score(E, etiquetas, metric="cosine")
    tamanos = sorted(pd.Series(etiquetas).value_counts().tolist(), reverse=True)
    print(k, round(sil, 3), tamanos)

4 0.067 [1013, 678, 446, 363]
6 0.094 [653, 621, 442, 346, 280, 158]
8 0.089 [549, 440, 356, 340, 305, 182, 180, 148]
10 0.1 [368, 346, 328, 327, 283, 255, 193, 151, 140, 109]
12 0.095 [345, 333, 278, 262, 252, 223, 188, 165, 138, 134, 102, 80]
15 0.094 [318, 233, 221, 217, 209, 193, 172, 163, 157, 139, 124, 121, 99, 72, 62]


### 4.2 Reading the groups with k=10

The silhouette scores for k greater than or equal to 6 give similar results (a 10% difference between them). k=10 is chosen because it sits in the middle of the other values tested, which gives a tolerance in the value of k on both sides of the one chosen. In this case, the distribution of the articles among the clusters is the most balanced.

For each group, the words that distinguish it (TF-IDF computed between groups) and the 3 headlines closest to its center, that is, the most representative ones, are shown.

In [31]:
def leer_clusters(k, n_titulos=3):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    etiquetas = km.fit_predict(E)
    dist = km.transform(E)

    docs = d.assign(c=etiquetas).groupby("c")["limpio"].apply(" ".join)
    tf = TfidfVectorizer(max_df=0.7)
    Xc = tf.fit_transform(docs)
    vocab = tf.get_feature_names_out()

    for i, c in enumerate(docs.index):
        idx = np.where(etiquetas == c)[0]
        centrales = idx[np.argsort(dist[idx, c])][:n_titulos]
        top = Xc[i].toarray()[0].argsort()[::-1][:8]
        print(f"\n=== Cluster {c} ({len(idx)} articles) ===")
        print("Words:", [vocab[j] for j in top])
        for j in centrales:
            print(" -", d["titulo"].iloc[j])
    return etiquetas

et10 = leer_clusters(10)
d["cluster_km"] = et10     


=== Cluster 0 (283 articles) ===
Words: ['película', 'komanche', 'conductora', 'musical', 'libro', 'festival', 'mamá', 'vestido']
 - De la Nahuala al Charro Negro: la saga “Las Leyendas” y su fórmula para dominar la taquilla nacional
 - El último mensaje de El Komanche se viraliza tras reportarse su muerte en México
 - Juls Janeiro evita mojarse en la guerra entre Beatriz Trapote y Belén Esteban pese a su apoyo público a su tía

=== Cluster 1 (368 articles) ===
Words: ['recomendar', 'clínico', 'proteína', 'salsa', 'grasa', 'queso', 'animal', 'humedad']
 - Aguas termales de Costa Rica podrían aliviar síntomas de osteoartritis y enfermedades de la piel, según especialista
 - Científicos argentinos diseñaron una proteína para tratar lesiones en la córnea tras una quemadura química
 - Cómo cuidar las hortensias cortadas para que duren más tiempo en el florero

=== Cluster 2 (327 articles) ===
Words: ['fiscalía', 'delito', 'policial', 'homicidio', 'crimen', 'juez', 'asesinato', 'alias']
 -

**Interpretation.** Reading the 3 headlines closest to the center of each group and the words that distinguish them, the 10 groups correspond to these topics:

| Group | Articles | Topic |
|---|---|---|
| Culture and entertainment | 283 | film, music, books, festivals, celebrity news |
| Health and everyday life | 368 | health, science, recipes, gardening |
| Crime and justice | 327 | crimes, homicides, prosecutors' offices, judges |
| Lottery and draws | 109 | lottery and draw results |
| Sports | 328 | matches and results |
| Elections in Brazil | 193 | runoff, Lula, Bolsonaro, and reactions |
| Weather and natural phenomena | 151 | rain, temperatures, earthquakes |
| International conflicts | 140 | Israel, Palestine, Ukraine, Iran |
| Economy | 255 | dollar, inflation, interest rates, housing |
| Politics | 346 | parties, elections in Spain and Argentina, Congress |

Observations:

- **The health and everyday life group is the most heterogeneous:** it brings together health, science, recipes, and gardening, and it is the largest in the corpus.
- **A single event dominates a group:** the Elections in Brazil group is made up almost entirely of articles about that runoff (result and reactions). It is a specific topic of the period and not a permanent topic of the newspaper. It also overlaps with Politics: an article about Argentine politicians who traveled to Brazil ended up in the latter.
- **The lottery group seems to include some sports article**, because the words `balón` and `bayern` appear among its terms. This was not verified.

**Limitations:** the reading was based on few headlines per group, so it is an impression and not a measurement. `k=10` has the highest silhouette (0.100), but the difference with `k=12` (0.095) and `k=8` (0.089) is minimal, so there is no clearly better value. The numbering of the groups depends on the seed and on the corpus.

### 4.3 BERTopic

The already computed embeddings (`E`) and the lemmatized text (`limpio`) are reused to describe each topic with its characteristic words. The main parameters are `min_cluster_size=30` (minimum number of articles to form a topic) and a fixed seed in UMAP so that the result is reproducible. Articles that HDBSCAN fails to assign to any topic end up in topic -1.

`min_cluster_size=30` and `min_samples=3` are set explicitly. With `min_cluster_size=10`, 82 topics came out, many of them a single event published in several articles. Combinations of both parameters were explored and this one was chosen because it gives 24 topics of even size, with 11% of articles without a topic. With `min_samples=5`, the same `min_cluster_size` gathered 514 articles into a single topic.

In [32]:
assert E.shape[0] == len(d)     # the rows of E and d must match

topic_model = BERTopic(
    embedding_model=modelo_emb,
    umap_model=UMAP(n_neighbors=10, n_components=5, min_dist=0.0,
                    metric="cosine", random_state=SEED),
    hdbscan_model=HDBSCAN(min_cluster_size=30, min_samples=3, prediction_data=True),
    vectorizer_model=CountVectorizer(min_df=2),
)
topics, probs = topic_model.fit_transform(d["limpio"].tolist(), embeddings=E)
d["topic"] = topics

info = topic_model.get_topic_info()
n_temas = (info["Topic"] != -1).sum()
n_out = (d["topic"] == -1).sum()
print(f"{n_temas} topics | {n_out} articles without topic ({n_out / len(d):.0%})")
print(info[["Topic", "Count", "Name"]].head(20).to_string())

24 topics | 271 articles without topic (11%)
    Topic  Count                                   Name
0      -1    271                     -1_no_año_él_dólar
1       0    270             0_partido_equipo_gol_punto
2       1    207                       1_no_él_año_vida
3       2    182                2_millón_no_empresa_año
4       3    182              3_no_gobierno_pp_elección
5       4    174      4_lula_bolsonaro_brasil_brasileño
6       5    169   5_no_investigación_presunto_judicial
7       6    147                   6_él_no_agua_persona
8       7    121           7_ataque_herido_ruso_militar
9       8     81      8_militar_dominicano_haití_fuerza
10      9     80                9_no_familia_policía_él
11     10     76                10_oro_balón_premio_año
12     11     60        11_sorteo_premio_ganador_número
13     12     59    12_temperatura_grado_clima_templado
14     13     58        13_aplicación_modelo_ia_usuario
15     14     42     14_colapinto_piloto_carrera_premio
16 

The topic names include filler words (`no`, `él`(he), `año`(year)), because the vectorizer that generates them does not discard frequent words. They are recalculated with `max_df=0.5`, which removes the words present in more than half of the topics. This only changes the names: the assignment of articles to topics is not modified.

In [33]:
vec_desc = CountVectorizer(min_df=2, max_df=0.5)
topic_model.update_topics(d["limpio"].tolist(), vectorizer_model=vec_desc)
print(topic_model.get_topic_info()[["Topic", "Count", "Name"]].to_string())

    Topic  Count                                          Name
0      -1    271             -1_dólar_personaje_película_vucic
1       0    270                    0_gol_liga_torneo_atlético
2       1    207           1_cantante_película_conductora_cine
3       2    182   2_trimestre_financiamiento_inflación_billón
4       3    182                   3_pp_vox_senado_candidatura
5       4    174                 4_lula_bolsonaro_flávio_silva
6       5    169                 5_juez_prisión_sexual_acusado
7       6    147                 6_animal_proteína_grasa_queso
8       7    121                7_ucraniano_drón_saudí_ucrania
9       8     81       8_dominicano_haití_salvadoreño_pandilla
10      9     80  9_streamer_homicidio_fallecimiento_asesinato
11     10     76                10_balón_kane_messi_futbolista
12     11     60                 11_sorteo_boleto_super_podrás
13     12     59      12_templado_cálido_centígrado_termómetro
14     13     58                    13_ia_apple_youtube

### 4.4 Comparison between KMeans and BERTopic

The 10 KMeans groups are cross-tabulated with the BERTopic topics to see whether the two methods agree. Each row is a BERTopic topic and each column a KMeans group; the number is the count of articles in that combination. If a topic falls almost entirely in one column, the two methods agree.

A clarification for reading the table: both methods use the same embeddings, so their agreement is expected and is not an independent validation. Topic -1 gathers the articles that BERTopic could not assign, and its name does not describe a topic.

In [34]:
# Numbering valid for SEED=42 and the current corpus.
# Numbering valid for SEED=42 and corpus_congelado.csv; review if either of them changes.
nombres_km = {
    0: "culture",
    1: "health/everyday life",
    2: "crime and justice",
    3: "lottery",
    4: "sports",
    5: "elections Brasil",
    6: "weather",
    7: "international conflicts",
    8: "economy",
    9: "politics",
}
d["group_km"] = d["cluster_km"].map(nombres_km)

nombres_t = topic_model.get_topic_info().set_index("Topic")["Name"]
ct = pd.crosstab(d["topic"], d["group_km"])[list(nombres_km.values())]
ct.index = [f"{t}: {nombres_t[t][:28]}" for t in ct.index]
print(ct.to_string())

group_km                          culture  health/everyday life  crime and justice  lottery  sports  elections Brasil  weather  international conflicts  economy  politics
-1: -1_dólar_personaje_película_       45                    44                 34        1       4                37       11                       17       54        24
0: 0_gol_liga_torneo_atlético           2                     3                  1        8     250                 0        0                        1        0         5
1: 1_cantante_película_conducto       190                     7                  3        2       1                 0        3                        0        0         1
2: 2_trimestre_financiamiento_i         0                    31                  1        0       0                 0        1                        1      146         2
3: 3_pp_vox_senado_candidatura          4                     3                 25        0       0                 1        1                   

**Reading the table.** The numbers come from the table; the BERTopic topics were interpreted from their keywords, without reading the articles.

- **The two methods agree on most topics:** 16 of the 24 BERTopic topics have at least 75% of their articles in a single KMeans group. For example, the goals and tournaments topic has 250 of its 270 articles in the sports group, the temperatures and weather topic has 58 of 59 in weather, and the Israel and Netanyahu topic has all 38 in international conflicts.
- **BERTopic splits the large groups into subtopics:** politics into several topics according to the country or event (PP and Vox, Venezuela, Mexico's INE, the Pope), sports into soccer and motorsport, conflicts into Israel and Ukraine, and weather into temperatures and earthquakes.
- **BERTopic isolates technology:** the topic on AI, Apple, YouTube, and Google has 57 of its 58 articles in the KMeans health and everyday life group, which is the most heterogeneous. BERTopic separates a topic that KMeans keeps mixed.
- **Eight topics are spread across several groups** (less than 75% in a single group):
  - The balón, Kane, and Messi topic (76 articles) is split between lottery (42) and sports (33). Hypothesis (not verified): the lottery group includes soccer pool results, since the words `balón` and `bayern` appear among its terms.
  - Three topics mix crime with other angles: the one on Dominicans, Haiti, and Salvadorans (38 in crime and 32 in politics, out of 81), the one on streamer and homicide (57 in crime and 23 in culture, out of 80), and the one on Ukraine, drones, and Saudi Arabia (80 of 121 in conflicts and 21 in crime).
  - Four topics (scholarship and IMSS, malbec and flood, pico y placa, capacity and holiday) are spread across three or four groups. By their keywords they seem to be local or one-off topics that do not match any KMeans group (unverified hypothesis).
- **The 271 articles without a topic (11%) are not evenly distributed:** they are 21% of the economy group, 19% of the elections in Brazil group, and 16% of the culture group, versus 1% in sports and in lottery. Hypothesis (not verified): groups with very specific vocabulary (sports, lottery) form dense areas that are easy to detect, whereas broader ones leave more articles unassigned.

### 4.5 Conclusion

- Embeddings make it possible to find thematic structure without labels in a corpus of 2500 articles: KMeans gives a general view of 10 groups and BERTopic a finer one of 24 topics, with 11% of articles unassigned.
- The two methods agree on the most well-defined topics: sports, weather, culture, Israel, and the elections in Brazil have almost all their articles in a single KMeans group. BERTopic also separates subtopics that KMeans keeps together, such as technology (AI, Apple, YouTube) within the health and everyday life group.
- The KMeans silhouettes are low (around 0.10). `k=10` is the maximum, but `k=8` (0.089) and `k=12` (0.094) are almost equivalent, so the groups are not clearly separated and `k` is not a firm choice.
- In BERTopic, `min_cluster_size` and `min_samples` had to be set explicitly. With `min_cluster_size=10`, 82 topics came out, many of them a single event published in several articles, and with `min_cluster_size=30` and `min_samples=5` a single topic gathered 514 articles. `min_cluster_size=30` and `min_samples=3` were chosen, which gives 24 topics of even size. At an earlier stage, with a smaller corpus (683 articles), the default value of `min_samples` also produced only 4 topics, one of which gathered 78% of the articles.

**Limitations:** the corpus covers approximately 3 days, so some groups are dominated by a single event published in several articles, such as the elections in Brazil. Both methods use the same embeddings, so their agreement is not an independent validation. The parameters were chosen by looking at the results, and the names of the KMeans groups were assigned by hand. The interpretation was based on a few headlines per group (KMeans) and on the keywords of each topic (BERTopic), so it is an impression and not a measurement.

## 5. Classifying by topic without training

**Task:** assign each article to a topic (politics, economy, sports, crime, culture, health, technology) without training any model on labeled examples.

**Labels for evaluation:** some Infobae URLs include a thematic section. The articles from those sections are used as the reference label, with this mapping:

| URL section | Topic |
|---|---|
| `politica` | politics |
| `economia` | economy |
| `deportes` | sports |
| `judiciales` | crime |
| `teleshow` | culture |
| `salud` | health |
| `tecno` | technology |

`judiciales` was assigned to `crime` by my own decision, and the same goes for `teleshow` and `culture`. `internacional` has no equivalent section and is left out of the evaluation. Articles from other sections (country editions, `noticias`, `sociedad`, etc.) have no thematic label and are not evaluated.

**Limitations:** the labels come from the outlet's own criteria and not from my own annotation, and the evaluation is only done on articles that the outlet already placed in a thematic section, which may be easier to classify than the rest of the corpus.

In [35]:
MAPA_SECCION = {
    "politica": "politics",
    "economia": "economy",
    "deportes": "sports",
    "judiciales": "crime",
    "teleshow": "culture",
    "salud": "health",
    "tecno": "technology",
}
d["topic_url"] = d["section"].map(MAPA_SECCION)
dt = d[d["topic_url"].notna()]

print(len(dt), "articles with a thematic label in the URL")
print(dt["topic_url"].value_counts())

312 articles with a thematic label in the URL
topic_url
sports        62
technology    58
health        50
culture       49
politics      39
economy       33
crime         21
Name: count, dtype: int64


### 5.1 Zero-shot with embeddings

The embedding of each article is compared with the embedding of a short description of each topic, and the closest topic is assigned (cosine similarity). Nothing is trained. The descriptions were written before seeing these results and are not tuned to these 312 articles, so that the evaluation is not biased by tuning. The model reads only the beginning of each article (~128 tokens).

In [36]:
DESCRIPCIONES = {
    "politics": "Noticia sobre política, gobierno y elecciones",
    "economy": "Noticia sobre economía, finanzas, precios y negocios",
    "sports": "Noticia sobre deportes, fútbol y competencias",
    "crime": "Noticia sobre crimen, violencia, atentados y justicia",
    "culture": "Noticia sobre espectáculos, cultura, premios y arte",
    "health": "Noticia sobre salud y medicina",
    "technology": "Noticia sobre tecnología, ciencia y medio ambiente",
}
nombres_tema = list(DESCRIPCIONES)
E_desc = modelo_emb.encode(list(DESCRIPCIONES.values()))

sims = util.cos_sim(E[dt.index.to_numpy()], E_desc)
y_true = dt["topic_url"]
y_pred = pd.Series([nombres_tema[i] for i in sims.argmax(dim=1).tolist()], index=dt.index)

print("baseline (most frequent):", round(y_true.value_counts(normalize=True).max(), 2))
print("accuracy:", round((y_pred == y_true).mean(), 2))
print("F1 macro:", round(f1_score(y_true, y_pred, average="macro"), 2))
print(classification_report(y_true, y_pred))

etq = sorted(DESCRIPCIONES)
print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=etq),
                   index=[f"real_{e}" for e in etq],
                   columns=[f"pred_{e}" for e in etq]))

baseline (most frequent): 0.2
accuracy: 0.67
F1 macro: 0.65
              precision    recall  f1-score   support

       crime       0.43      0.57      0.49        21
     culture       0.56      0.47      0.51        49
     economy       0.65      0.73      0.69        33
      health       0.76      0.74      0.75        50
    politics       0.63      0.92      0.75        39
      sports       0.84      0.85      0.85        62
  technology       0.65      0.41      0.51        58

    accuracy                           0.67       312
   macro avg       0.64      0.67      0.65       312
weighted avg       0.67      0.67      0.66       312

                 pred_crime  pred_culture  pred_economy  pred_health  \
real_crime               12             0             4            2   
real_culture              9            23             0            5   
real_economy              1             0            24            0   
real_health               4             0             1

In [37]:
def ver_errores(real, pred, n=10):
    m = (y_true == real) & (y_pred == pred)
    print(f"--- real={real}, predicted={pred}: {m.sum()} articles ---")
    for t in dt.loc[m, "titulo"].head(n):
        print(" -", t)

ver_errores("technology", "culture")
ver_errores("health", "technology")
ver_errores("technology", "politics")

--- real=technology, predicted=culture: 15 articles ---
 - El director de Google Argentina habló sobre YouTube: “La gente es la que hoy dicta el prime time en TV”
 - Cómo tener karaoke gratis en casa con inteligencia artificial: el truco para quitar la voz
 - Los dos juegos de terror que Epic Games Store regala hasta el 29 de octubre
 - Los rumores de Half-Life 3 vuelven: el capítulo de la saga de Valve más esperado
 - Las guerreras K-pop en Halloween: cuál es el disfraz más buscado, según Google
 - YouTube en Argentina: los 10 videos que son tendencia hoy
 - YouTube en Chile: la lista de los 10 videos más reproducidos que son tendencia hoy
 - YouTube en Colombia: los 10 videos que son tendencia hoy
 - Xuper TV APK v7.2.6 Descarga Gratis (Oficial) 2026: por qué instalar esta app es la peor decisión
 - Elon Musk prometió una película completa sobre ‘La Odisea’ hecha con IA y mostró apenas siete minutos
--- real=health, predicted=technology: 8 articles ---
 - Por qué comer siempre a la m

**Interpretation.** The method gets 209 of 312 articles right (accuracy 0.67, macro F1 0.65), versus 0.20 for the baseline of always predicting `deportes`. Sports (precision 0.84, recall 0.85) and health (0.76 and 0.74) are classified best. Politics has high recall (0.92) but low precision (0.63): it attracts articles from other categories. Technology (recall 0.41) and culture (0.47) lose more than half of their articles.

The four largest errors, read from the headlines (not from the full text):

- **Technology → culture (15 articles, 26% of technology; 10 were read):** almost all deal with YouTube, video games, or AI-generated content. The outlet's technology section includes this content and the description of `cultura` overlaps with it: it is a boundary problem between categories rather than a clear error of the method.
- **Culture → crime (9 articles):** four are the same event (a complaint about mistreatment), two deal with threats or harassment, and three have no apparent criminal content and look like errors of the method.
- **Health → technology (8 articles):** five are studies or science stories, and the description of `tecnología` includes "science", which would explain part of the confusion (hypothesis, not verified). Two are health tips without science and look like errors of the method, and one (decoration) does not fit any category.
- **Technology → politics (7 articles):** two have political content (a donation to the legislative elections and a law on AI-related notices). The other five have no apparent political content from the headline and look like errors of the method.

**Limitations:** the labels come from the URL section, that is, from the outlet's own criteria, and not from my own annotation. Only the 312 articles placed in a thematic section are evaluated, which may be easier than the rest of the corpus. The same event can appear in several articles, which weighs heavily in small categories such as `crimen` (21 articles). The topic descriptions were not tuned to these articles; the one for `tecnología` includes "science and environment", which overlaps with health.

## 6. General conclusions

**Corpus:** 2500 articles from the Infobae RSS feed, published between October 23, 2026 (19:00 UTC) and October 27, 2026 (03:27 UTC), that is, about 80 hours.

**Main results**

| Task | Method | Result |
|---|---|---|
| Predict the edition (5 classes, 1852 articles) | Baseline: always `america` | accuracy 0.42 |
| | TF-IDF + logistic regression, full article | macro F1 0.85 |
| | TF-IDF + logistic regression, first 90 words | macro F1 0.79 |
| | Embeddings + logistic regression | macro F1 0.65 |
| Discover topics without labels | KMeans | k=10 (silhouette 0.100) |
| | BERTopic | 24 topics, 271 articles without a topic (11%) |
| Classify by topic without training (7 classes, 312 articles) | Baseline: always `deportes` | accuracy 0.20 |
| | Zero-shot with embeddings | accuracy 0.67, macro F1 0.65 |

**What was learned**

1. **The most useful method depends on the task.** To predict the edition, TF-IDF outperformed embeddings by 20 points of macro F1 with the full article (0.85 versus 0.65) and by 14 with text of comparable length (0.79 versus 0.65). Hypothesis (not verified): the edition is given away by specific words (places, politicians, institutions) that TF-IDF sees directly, whereas an embedding summarizes the topic. The main confusion was between `america` and `espana` (203 of the 313 errors, 65%).
2. **Embeddings made it possible to explore and classify by topic without training.** KMeans and BERTopic agree on most topics (16 of 24 BERTopic topics have at least 75% of their articles in a single KMeans group), and the zero-shot got 67% of the articles right versus 20% for the baseline. In these two tasks, no comparison against TF-IDF was made.
3. **Controls and default values matter.** Trimming the articles to 90 words separated how much of TF-IDF's advantage came from text length (6 points) and how much from the method (14 points). In BERTopic, `min_cluster_size=10` produced 82 topics, many of them a single event published in several articles. With `min_cluster_size=30` and `min_samples=5`, a single topic gathered 514 articles, and at an earlier stage, with 683 articles, the default value of `min_samples` gave only 4 topics.
4. **Reading the errors changes their interpretation.** In the zero-shot, part of the errors are not clear failures of the method: technology → culture (15 articles) mixes YouTube, video games, and AI content, which sit on the boundary between both categories, and four of the nine culture → crime articles are the same event (a complaint about mistreatment).

**Limitations**

- The corpus covers about 80 hours. Several topics correspond to a single event published in several articles (a KMeans group of 193 articles deals almost only with the Brazil runoff), and those articles can end up split between training and test in cross-validation, which can inflate the results.
- The feed only delivers the latest articles, so the periods in which the script was not run remain uncovered.
- The hyperparameters (`C`, `class_weight`, `k`, `min_cluster_size`, `min_samples`) were chosen by looking at results from the same corpus, so the numbers may be somewhat optimistic.
- The labels in section 5 come from the URL section (the outlet's own criteria, not my own annotation), cover only 312 articles, and `crimen` has barely 21.
- KMeans, BERTopic, and the zero-shot use the same embeddings, which read only the first ~128 tokens of each article. Their agreement is not an independent validation.
- The interpretation of the groups was based on a few headlines per group and on keywords: it is a qualitative reading, not a measurement.

